# 🐱 Cat vs Dog Classifier — Complete CNN Guide

**Role:** Computer Vision Engineer  
**Task:** Build a binary image classifier from scratch using Convolutional Neural Networks

---

## Why is this harder than MNIST?

| | MNIST | Cat vs Dog |
|---|---|---|
| Image size | 28×28 px | 150×150 px |
| Colour | Grayscale (1 channel) | RGB (3 channels) |
| Subject position | Always centred | Anywhere in frame |
| Background | Always black | Varied, complex |
| Variation | Just different handwriting | Breeds, poses, lighting, occlusion |
| Difficulty | Easy (~99% acc) | Harder (~85–90% acc with small data) |

---

## What a CNN learns layer by layer

```
Layer 1  →  edges, colour gradients ("there is a dark-to-light boundary here")
Layer 2  →  textures, corners       ("these edges form a fur texture")
Layer 3  →  parts                   ("these textures form a pointed ear")
Layer 4  →  high-level structures   ("this arrangement of parts = cat face")
Dense    →  final decision          ("output 0.05 → cat, 95% confident")
```

---

## Full pipeline

```
Raw images on disk
      ↓
Preprocessing (resize + normalise)
      ↓
Data Augmentation (flip, rotate, zoom, shift)
      ↓
CNN Architecture (Conv → Pool → Conv → Pool → Dense)
      ↓
Training (RMSprop + binary_crossentropy)
      ↓
Evaluation (accuracy, loss curves, confusion matrix)
      ↓
Prediction (sigmoid output → cat or dog + confidence)
```

## Setup — install dependencies

In [ ]:
# Uncomment to install if needed
# !pip install tensorflow numpy matplotlib scikit-learn

In [ ]:
import os
import zipfile
import urllib.request
import shutil

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image as keras_image

# Fix random seeds for reproducibility
# Without this, results change every run due to random weight initialisation
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU present : {len(tf.config.list_physical_devices('GPU')) > 0}")

---

# Step 1 — Dataset Download & Directory Setup

## Theory

We use the **Kaggle Dogs vs Cats** subset (2,000 images) provided by Google's ML education team.

Why a subset of the full 25,000-image dataset?
- Training 25k images on CPU takes hours
- 2,000 images teaches all the same CNN concepts
- The cost is slightly lower accuracy (~85% vs ~95%)

## Directory structure

Keras's `ImageDataGenerator.flow_from_directory()` reads class labels from **folder names** automatically.
So you just need to put cat images in a `cats/` folder and dog images in a `dogs/` folder:

```
data/
├── train/
│   ├── cats/    ← 1000 cat training images
│   └── dogs/    ← 1000 dog training images
└── validation/
    ├── cats/    ← 500 cat validation images
    └── dogs/    ← 500 dog validation images
```

No manual labelling needed — the folder name IS the label.

In [ ]:
# ── Directory paths ───────────────────────────────────────────────────────────
BASE_DIR            = 'data'
TRAIN_DIR           = os.path.join(BASE_DIR, 'train')
VALIDATION_DIR      = os.path.join(BASE_DIR, 'validation')
TRAIN_CATS_DIR      = os.path.join(TRAIN_DIR,      'cats')
TRAIN_DOGS_DIR      = os.path.join(TRAIN_DIR,      'dogs')
VALIDATION_CATS_DIR = os.path.join(VALIDATION_DIR, 'cats')
VALIDATION_DOGS_DIR = os.path.join(VALIDATION_DIR, 'dogs')

# Create all directories (exist_ok=True means no error if they already exist)
for d in [TRAIN_CATS_DIR, TRAIN_DOGS_DIR, VALIDATION_CATS_DIR, VALIDATION_DOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Directory structure ready')
print(f'  {TRAIN_CATS_DIR}/')
print(f'  {TRAIN_DOGS_DIR}/')
print(f'  {VALIDATION_CATS_DIR}/')
print(f'  {VALIDATION_DOGS_DIR}/')

In [ ]:
# ── Download and organise dataset ────────────────────────────────────────────
zip_path = 'cats_and_dogs_filtered.zip'
url      = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'

# Download only if not already downloaded
if not os.path.exists(zip_path):
    print('Downloading dataset (~60 MB)...')
    urllib.request.urlretrieve(url, zip_path)
    print('✓ Download complete')
else:
    print('✓ Already downloaded')

# Extract and copy only if images not already in place
if len(os.listdir(TRAIN_CATS_DIR)) == 0:
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')   # extracts to ./cats_and_dogs_filtered/

    src = 'cats_and_dogs_filtered'
    for fname in sorted(os.listdir(os.path.join(src, 'train', 'cats')))[:1000]:
        shutil.copy(os.path.join(src, 'train', 'cats', fname), TRAIN_CATS_DIR)
    for fname in sorted(os.listdir(os.path.join(src, 'train', 'dogs')))[:1000]:
        shutil.copy(os.path.join(src, 'train', 'dogs', fname), TRAIN_DOGS_DIR)
    for fname in sorted(os.listdir(os.path.join(src, 'validation', 'cats')))[:500]:
        shutil.copy(os.path.join(src, 'validation', 'cats', fname), VALIDATION_CATS_DIR)
    for fname in sorted(os.listdir(os.path.join(src, 'validation', 'dogs')))[:500]:
        shutil.copy(os.path.join(src, 'validation', 'dogs', fname), VALIDATION_DOGS_DIR)
    print('✓ Images organised')
else:
    print('✓ Images already in place')

# Report counts
print(f'\nTrain cats      : {len(os.listdir(TRAIN_CATS_DIR)):,}')
print(f'Train dogs      : {len(os.listdir(TRAIN_DOGS_DIR)):,}')
print(f'Validation cats : {len(os.listdir(VALIDATION_CATS_DIR)):,}')
print(f'Validation dogs : {len(os.listdir(VALIDATION_DOGS_DIR)):,}')

In [ ]:
# ── Visualise sample images ───────────────────────────────────────────────────
# Look at real examples before building the model.
# Notice: varied sizes, backgrounds, poses, lighting — this is what makes
# the problem genuinely challenging.

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Sample training images — Top row: Cats, Bottom row: Dogs', fontsize=13)

cat_files = os.listdir(TRAIN_CATS_DIR)[:5]
dog_files = os.listdir(TRAIN_DOGS_DIR)[:5]

for i, fname in enumerate(cat_files):
    img = mpimg.imread(os.path.join(TRAIN_CATS_DIR, fname))
    axes[0][i].imshow(img)
    axes[0][i].set_title(f'Cat — {img.shape[0]}×{img.shape[1]}', fontsize=9)
    axes[0][i].axis('off')

for i, fname in enumerate(dog_files):
    img = mpimg.imread(os.path.join(TRAIN_DOGS_DIR, fname))
    axes[1][i].imshow(img)
    axes[1][i].set_title(f'Dog — {img.shape[0]}×{img.shape[1]}', fontsize=9)
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()

print('Notice: images have different sizes → we must resize to a fixed size before training')

---

# Step 2 — Image Preprocessing

## Theory

Raw images cannot be fed directly into a neural network. Three problems to solve:

**Problem 1 — Variable size**  
A CNN has fixed-size weight matrices. A 2000×1500 photo and a 640×480 photo can't both fit the same input layer. We must resize every image to the same fixed dimensions: **150×150 pixels**.

**Problem 2 — Pixel scale**  
Raw pixels are integers in [0, 255]. Large values → large gradients → unstable training.  
We normalise by dividing by 255.0, mapping to [0.0, 1.0].

**Problem 3 — Colour channels**  
Each colour image has 3 channels (Red, Green, Blue). Shape: `(height, width, 3)`.  
We keep all 3 — colour helps (golden retriever has distinctive golden colour, cats often grey).

## Mathematics

**Normalisation:**
$$X_{\text{norm}} = \frac{X}{255.0} \quad \Rightarrow \quad X_{\text{norm}} \in [0.0,\; 1.0]$$

**Input tensor after preprocessing:**
$$X \in \mathbb{R}^{150 \times 150 \times 3} \quad \text{(67,500 values per image)}$$

**Why CNNs not MLPs for this?**  
A single Dense layer from 67,500 inputs to 512 neurons needs $67{,}500 \times 512 = 34.5M$ parameters.  
CNNs use weight sharing — the same 3×3 filter slides everywhere, so 32 filters only need $3 \times 3 \times 3 \times 32 = 864$ parameters for the first layer!

In [ ]:
# ── Global constants ──────────────────────────────────────────────────────────
IMG_HEIGHT = 150    # resize all images to this height in pixels
IMG_WIDTH  = 150    # resize all images to this width in pixels
BATCH_SIZE = 32     # number of images processed per weight update
                    # 32 is a sweet spot: fits in RAM, good gradient estimate

# ── Validation data generator ─────────────────────────────────────────────────
# IMPORTANT: validation data is NEVER augmented — we want real images
# to get an honest estimate of model performance on unseen data.
validation_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255    # divides every pixel by 255 → [0,1]
)

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,                         # reads from data/validation/
    target_size=(IMG_HEIGHT, IMG_WIDTH),    # resize every image to 150×150
    batch_size=BATCH_SIZE,
    class_mode='binary'                     # 'binary' = 2 classes
                                            # returns labels as 0.0 or 1.0
                                            # use 'categorical' for 3+ classes
)

print(f'Class mapping: {validation_generator.class_indices}')
# → {'cats': 0, 'dogs': 1}  Keras assigns labels alphabetically by folder name

# Inspect one batch
images, labels = next(validation_generator)
print(f'\nOne batch:')
print(f'  images shape : {images.shape}   ← (batch, height, width, channels)')
print(f'  labels shape : {labels.shape}   ← (batch,) — one float per image')
print(f'  pixel range  : {images.min():.2f} – {images.max():.2f}  ← normalised ✓')
print(f'  labels sample: {labels[:8]}  ← 0=cat, 1=dog')

---

# Step 3 — Data Augmentation

## Theory

With only 2,000 training images, our CNN risks **overfitting** — memorising the training images instead of learning general patterns.

**Data augmentation** artificially expands the training set by applying random transformations each time an image is loaded. The model sees a slightly different version of each image on every epoch.

Think about it: if you've only seen cats photographed from the left, you might not recognise a cat photographed from the right. Augmentation ensures the model sees cats and dogs in all orientations, scales, and positions.

## Mathematics

**Horizontal flip:**  
$$X_{\text{flip}}[i,\; j] = X[i,\; W-1-j]$$

**Rotation by angle θ** (around image centre $(c_x, c_y)$):  
$$\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x - c_x \\ y - c_y \end{bmatrix} + \begin{bmatrix} c_x \\ c_y \end{bmatrix}$$

**Zoom (scale factor $s$):**  
$$X_{\text{zoom}}[i,\; j] = X\!\left[\frac{i}{s},\; \frac{j}{s}\right]$$

**Key rule:** Augmentation is applied **only to training data**, never to validation or test data.

In [ ]:
# ── Training data generator WITH augmentation ─────────────────────────────────
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,          # normalise: [0,255] → [0.0,1.0]

    # ── Geometric augmentations ──────────────────────────────────────────────
    rotation_range=40,       # randomly rotate 0° to 40° each time
                             # Why 40°? Enough variation without making
                             # the image unrecognisable.

    width_shift_range=0.2,   # shift left/right by up to 20% of image width
                             # cats/dogs aren't always centred

    height_shift_range=0.2,  # shift up/down by up to 20% of image height

    shear_range=0.2,         # shear angle in radians
                             # creates a 'slanted' version of the image

    zoom_range=0.2,          # zoom in or out by up to 20%
                             # animals appear at different distances

    horizontal_flip=True,    # randomly mirror the image left-right
                             # a dog facing left = dog facing right = still a dog
                             # DON'T use vertical_flip for natural photos!
                             # (upside-down dogs are unrealistic)

    fill_mode='nearest'      # after rotating/shifting, some pixels are 'empty'.
                             # 'nearest': fill with the closest existing pixel.
                             # Other options: 'constant' (black), 'reflect', 'wrap'
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print(f'Training samples   : {train_generator.samples:,}')
print(f'Validation samples : {validation_generator.samples:,}')
print(f'Batch size         : {BATCH_SIZE}')
print(f'Steps per epoch    : {train_generator.samples // BATCH_SIZE}')

In [ ]:
# ── Visualise augmentations ───────────────────────────────────────────────────
# Show 6 different random augmentations of the SAME image.
# This is crucial for sanity-checking that augmentations look realistic.

def show_augmentations(img_path, n=6):
    img       = keras_image.load_img(img_path)
    img_array = keras_image.img_to_array(img)
    img_array = img_array.reshape((1,) + img_array.shape)  # (1, H, W, 3)

    aug = keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40, width_shift_range=0.2, height_shift_range=0.2,
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
        fill_mode='nearest'
    )

    fig, axes = plt.subplots(1, n+1, figsize=(3*(n+1), 3))
    fig.suptitle('Original image (left) vs 6 augmented versions (right)', fontsize=11)

    # Show original
    axes[0].imshow(keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH)))
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    for i, batch in enumerate(aug.flow(img_array, batch_size=1)):
        axes[i+1].imshow(batch[0].astype('uint8'))
        axes[i+1].set_title(f'Aug {i+1}', fontsize=9)
        axes[i+1].axis('off')
        if i == n-1:
            break

    plt.tight_layout()
    plt.show()

first_cat = os.path.join(TRAIN_CATS_DIR, os.listdir(TRAIN_CATS_DIR)[0])
first_dog = os.path.join(TRAIN_DOGS_DIR, os.listdir(TRAIN_DOGS_DIR)[0])

show_augmentations(first_cat)
show_augmentations(first_dog)

---

# Step 4 — Convolution, Filters & Pooling — Deep Dive

## Theory: Convolution

A convolution is a **sliding dot product** between a small matrix (the **filter** or **kernel**) and a local patch of the image.

The filter slides across the entire image one position at a time. At every position it:
1. Multiplies each filter value with the corresponding pixel
2. Sums all those products
3. Writes the result to an output matrix called the **feature map**

## Mathematics: One convolution step

Image patch $P$ (3×3):
$$P = \begin{bmatrix} 10 & 20 & 30 \\ 40 & 50 & 60 \\ 70 & 80 & 90 \end{bmatrix}$$

Vertical edge detection filter $W$ (3×3):
$$W = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix}$$

Output value:
$$Z = \sum_{m,n} P[m,n] \cdot W[m,n] = (-10+0+30) + (-40+0+60) + (-70+0+90) = 60$$

Strong positive response → **vertical edge detected here!**

## Feature map size formula

$$\text{Output size} = \frac{H - F + 2P}{S} + 1$$

where $H$ = input size, $F$ = filter size, $P$ = padding, $S$ = stride.

## Theory: MaxPooling

MaxPooling2D(2,2) takes the **maximum value** in every 2×2 window:

$$\text{Pool}\!\left(\begin{bmatrix} 0.8 & 0.2 \\ 0.1 & 0.9 \end{bmatrix}\right) = 0.9$$

**Why max?** We care about *whether* a feature was detected, not *exactly where* in the 2×2 block. This gives **translation invariance** — a cat's ear shifted 1 pixel is still detected.

In [ ]:
# ── Manual convolution demo ───────────────────────────────────────────────────
# Apply classic hand-crafted filters to an image to build intuition
# for what convolutions do BEFORE the CNN learns its own filters.

from scipy import ndimage   # for applying filters manually

# Load a sample image and convert to grayscale for easy visualisation
sample_path = os.path.join(TRAIN_CATS_DIR, os.listdir(TRAIN_CATS_DIR)[0])
img_rgb     = keras_image.load_img(sample_path, target_size=(150, 150))
img_gray    = np.array(img_rgb.convert('L'))   # L = Luminance (grayscale)

# Classic hand-crafted filters
vertical_edge   = np.array([[-1,0,1], [-1,0,1], [-1,0,1]])
horizontal_edge = np.array([[-1,-1,-1], [0,0,0], [1,1,1]])
blur            = np.ones((5,5)) / 25           # average filter
sharpen         = np.array([[0,-1,0], [-1,5,-1], [0,-1,0]])

# Apply each filter
vertical_map   = ndimage.convolve(img_gray.astype(float), vertical_edge)
horizontal_map = ndimage.convolve(img_gray.astype(float), horizontal_edge)
blur_map       = ndimage.convolve(img_gray.astype(float), blur)
sharpen_map    = ndimage.convolve(img_gray.astype(float), sharpen)

# Display
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Manual filter application — understanding what convolutions do', fontsize=12)

axes[0].imshow(img_gray, cmap='gray');        axes[0].set_title('Original (grayscale)')
axes[1].imshow(np.abs(vertical_map),   cmap='gray'); axes[1].set_title('Vertical edges\n[-1,0,1]')
axes[2].imshow(np.abs(horizontal_map), cmap='gray'); axes[2].set_title('Horizontal edges\n[-1,-1,-1],[0,0,0],[1,1,1]')
axes[3].imshow(blur_map,   cmap='gray');      axes[3].set_title('Blur\n(5×5 average)')
axes[4].imshow(np.clip(sharpen_map, 0, 255), cmap='gray'); axes[4].set_title('Sharpen\n[0,-1,0],[-1,5,-1],[0,-1,0]')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Key insight: In a CNN, we do NOT design these filters by hand.')
print('The CNN LEARNS the optimal filter values through backpropagation!')

In [ ]:
# ── MaxPooling demo ───────────────────────────────────────────────────────────
# Manually demonstrate what MaxPool2D(2,2) does to a feature map

print('MaxPooling2D(2,2) demo\n')
print('Input feature map (4×4):')
feature_map = np.array([
    [1,  3,  2,  4],
    [5,  6,  1,  2],
    [7,  1,  9,  3],
    [2,  8,  4,  6]
])
print(feature_map)

print('\nMaxPooling takes the MAX in each 2×2 non-overlapping window:')
print('  Top-left    [1,3,5,6]  → max = 6')
print('  Top-right   [2,4,1,2]  → max = 4')
print('  Bottom-left [7,1,2,8]  → max = 8')
print('  Bottom-right[9,3,4,6]  → max = 9')

output_pool = np.array([[6, 4], [8, 9]])
print('\nOutput (2×2):')
print(output_pool)
print(f'\nSize reduction: {feature_map.shape} → {output_pool.shape} (halved in each dimension)')

---

# Step 5 — CNN Architecture

## Theory

### Design principle: progressive deepening

We start with **few filters** for simple features and **double the filter count** at each block as the features become more complex. Meanwhile, MaxPooling **halves** the spatial dimensions, so computation stays manageable.

```
Block     Filters   Spatial size   What it learns
─────────────────────────────────────────────────────
Input     —         150×150×3      Raw RGB pixels
Block 1   32        74×74×32       Edges, colour gradients
Block 2   64        36×36×64       Textures, corners, fur patterns
Block 3   128       17×17×128      Object parts (ears, snouts, eyes)
Block 4   128       7×7×128        Whole face/body structures
Dense(512) —        512            Abstract feature combinations
Output    —         1              P(dog) — probability it's a dog
```

### Why sigmoid + 1 neuron for binary classification?

$$\sigma(x) = \frac{1}{1 + e^{-x}} \in (0, 1)$$

- Output < 0.5 → **cat** (confidence = $(1 - \text{output}) \times 100\%$)
- Output ≥ 0.5 → **dog** (confidence = $\text{output} \times 100\%$)

### Loss function: Binary Cross-Entropy

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

Perfect prediction: $\hat{y}=1, y=1$ → $\mathcal{L} = -\log(1) = 0$ ✓  
Worst prediction: $\hat{y} \approx 0, y=1$ → $\mathcal{L} = -\log(\epsilon) \to \infty$ ✓

In [ ]:
def build_cat_dog_cnn():
    """
    Builds the Cat vs Dog CNN.

    Architecture:
        Input  (150, 150, 3)
        Block1: Conv2D(32)  + MaxPool  → (74, 74, 32)
        Block2: Conv2D(64)  + MaxPool  → (36, 36, 64)
        Block3: Conv2D(128) + MaxPool  → (17, 17, 128)
        Block4: Conv2D(128) + MaxPool  → ( 7,  7, 128)
        Flatten                        → (6272,)
        Dropout(0.5)
        Dense(512, relu)
        Dense(1,   sigmoid)            → P(dog)
    """
    model = models.Sequential([

        # ── Block 1: Detect basic edges ───────────────────────────────────────
        # 32 filters, 3×3 kernel.
        # 'valid' padding (default): output shrinks by 2 on each side.
        # Input  (150,150,3)  → after Conv  (148,148,32) → after Pool (74,74,32)
        layers.Conv2D(32, (3,3), activation='relu',
                      input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), name='conv1'),
        layers.MaxPooling2D(2, 2, name='pool1'),

        # ── Block 2: Detect textures ──────────────────────────────────────────
        # 64 filters — more than block 1 because layer 2 combines
        # simple patterns from block 1 into more complex ones.
        # Input (74,74,32) → after Conv (72,72,64) → after Pool (36,36,64)
        layers.Conv2D(64, (3,3), activation='relu', name='conv2'),
        layers.MaxPooling2D(2, 2, name='pool2'),

        # ── Block 3: Detect object parts ─────────────────────────────────────
        # 128 filters — deeper layers need more capacity to represent
        # complex combinations of textures.
        # Input (36,36,64) → after Conv (34,34,128) → after Pool (17,17,128)
        layers.Conv2D(128, (3,3), activation='relu', name='conv3'),
        layers.MaxPooling2D(2, 2, name='pool3'),

        # ── Block 4: Detect high-level structures ─────────────────────────────
        # Still 128 filters — spatial size is now small (17×17) but
        # each of the 128 feature maps encodes rich information.
        # Input (17,17,128) → after Conv (15,15,128) → after Pool (7,7,128)
        layers.Conv2D(128, (3,3), activation='relu', name='conv4'),
        layers.MaxPooling2D(2, 2, name='pool4'),

        # ── Transition: 3D → 1D ──────────────────────────────────────────────
        # Flatten: (7, 7, 128) = 6,272 values → 1D vector of 6,272
        # Dense layers need 1D input.
        layers.Flatten(name='flatten'),

        # ── Regularisation ───────────────────────────────────────────────────
        # Dropout(0.5): randomly zero 50% of activations during training.
        # This forces redundant representations → prevents overfitting.
        # At prediction time, Dropout is OFF (all neurons active, values ×0.5)
        layers.Dropout(0.5, name='dropout'),

        # ── Dense classifier ─────────────────────────────────────────────────
        # 512 neurons combine extracted features into a high-level representation.
        layers.Dense(512, activation='relu', name='dense1'),

        # ── Output ───────────────────────────────────────────────────────────
        # sigmoid: 1/(1+e^{-x}) maps any real number to (0,1)
        # output ≥ 0.5 → dog,  output < 0.5 → cat
        layers.Dense(1, activation='sigmoid', name='output')
    ])
    return model


model = build_cat_dog_cnn()
model.summary()

In [ ]:
# ── Understand parameter count ────────────────────────────────────────────────
print('Parameter count breakdown by layer:\n')
print(f'{"Layer":<20} {"Output Shape":<22} {"Parameters":>12}')
print('-' * 56)

for layer in model.layers:
    params     = layer.count_params()
    out_shape  = str(layer.output_shape)
    print(f'{layer.name:<20} {out_shape:<22} {params:>12,}')

print('-' * 56)
print(f'{"TOTAL":<20} {"":<22} {model.count_params():>12,}')
print(f'\nMemory footprint (float32): ~{model.count_params()*4/1024/1024:.1f} MB')

---

# Step 6 — Compile & Train

## Theory: RMSprop Optimiser

We use **RMSprop** — well suited for CNNs on image data.

RMSprop maintains a per-parameter running average of squared gradients, which normalises the update:

$$v_t = \rho \cdot v_{t-1} + (1-\rho) \cdot g_t^2$$

$$W_t = W_{t-1} - \frac{\alpha}{\sqrt{v_t + \varepsilon}} \cdot g_t$$

Effect: parameters with large gradients get small updates (stable). Parameters with small gradients get relatively larger updates (faster learning).

Default: $\alpha = 10^{-4}$, $\rho = 0.9$, $\varepsilon = 10^{-7}$

## Theory: Training loop

```
FOR each epoch (1 to 30):
  FOR each batch of 32 images:
    1. Forward pass  → compute sigmoid output ŷ
    2. Loss          → binary_crossentropy(y, ŷ)
    3. Backward pass → ∂L/∂W for every weight (chain rule)
    4. RMSprop       → update all weights
  Report: train_loss, train_acc, val_loss, val_acc
```

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────────
model.compile(
    # RMSprop with lr=1e-4 (small learning rate for stable training on images)
    # Rule of thumb: start 1e-3, reduce to 1e-4 if oscillating loss
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),

    # binary_crossentropy: correct loss when output layer uses sigmoid
    # and problem has exactly 2 classes.
    loss='binary_crossentropy',

    metrics=['accuracy']   # easy human-readable metric
)
print('Model compiled.')

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = 'best_cat_dog_model.keras'

callbacks = [
    # Save the model whenever val_accuracy improves.
    # If val_acc goes 0.70→0.72→0.71, only the 0.72 checkpoint is kept.
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),

    # If val_loss doesn't improve for 5 consecutive epochs:
    # new learning rate = current_lr × 0.5
    # This helps the optimizer take smaller steps to escape a plateau.
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    ),

    # Stop training if val_accuracy doesn't improve for 10 epochs.
    # restore_best_weights=True: automatically reload best saved weights.
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    )
]

print('Callbacks registered:')
for cb in callbacks:
    print(f'  • {cb.__class__.__name__}')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
NUM_EPOCHS        = 30
STEPS_PER_EPOCH   = train_generator.samples      // BATCH_SIZE
VALIDATION_STEPS  = validation_generator.samples // BATCH_SIZE

print(f'Training on {train_generator.samples:,} images')
print(f'Steps per epoch : {STEPS_PER_EPOCH}  (= {train_generator.samples} ÷ {BATCH_SIZE})')
print(f'Max epochs      : {NUM_EPOCHS}')
print()

history = model.fit(
    train_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=NUM_EPOCHS,
    validation_data=validation_generator,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks,
    verbose=1
)

---

# Step 7 — Evaluation

## Reading the training curves

| Pattern | Meaning | Fix |
|---|---|---|
| Train acc >> Val acc | Overfitting | More augmentation, more dropout, fewer filters |
| Both accuracies low | Underfitting | More epochs, more filters, less dropout |
| Val loss rising while train loss falls | Overfitting | Reduce model capacity |
| Both curves converge | Good fit ✓ | — |

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'b-', linewidth=2, label='Train accuracy')
ax1.plot(epochs, val_acc, 'r--', linewidth=2, label='Validation accuracy')
ax1.axhline(y=max(val_acc), color='gray', linestyle=':', alpha=0.7,
            label=f'Best val: {max(val_acc):.3f}')
ax1.set_title('Accuracy', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, loss,     'b-', linewidth=2, label='Train loss')
ax2.plot(epochs, val_loss, 'r--', linewidth=2, label='Validation loss')
ax2.axhline(y=min(val_loss), color='gray', linestyle=':', alpha=0.7,
            label=f'Best val: {min(val_loss):.3f}')
ax2.set_title('Loss', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Binary cross-entropy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=15)
plt.tight_layout()
plt.show()

print(f'Best validation accuracy : {max(val_acc)*100:.1f}%')
print(f'Stopped at epoch         : {len(acc)}')

In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────
model.load_weights(CHECKPOINT_PATH)   # reload best checkpoint
val_loss_f, val_acc_f = model.evaluate(validation_generator,
                                        steps=VALIDATION_STEPS, verbose=0)
print(f'Validation accuracy : {val_acc_f:.4f}  ({val_acc_f*100:.1f}%)')
print(f'Validation loss     : {val_loss_f:.4f}')

In [ ]:
# ── Visualise what the CNN learned ───────────────────────────────────────────
# Show the filter weights and feature maps from the first Conv layer.

# Filter weights
conv1_weights = model.get_layer('conv1').get_weights()[0]  # (3,3,3,32)
print(f'First Conv layer weights shape: {conv1_weights.shape}')
print(f'  = (kernel_h={conv1_weights.shape[0]}, kernel_w={conv1_weights.shape[1]},'
      f' input_channels={conv1_weights.shape[2]}, num_filters={conv1_weights.shape[3]})')

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('32 learned filter weights from Conv Layer 1\n'
             '(Each 3×3 square = one learned filter)', fontsize=12)

for i, ax in enumerate(axes.flat):
    f = conv1_weights[:, :, :, i]                       # (3,3,3) — RGB filter
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)     # normalise for display
    ax.imshow(f)
    ax.set_title(f'F{i+1}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature maps visualisation ────────────────────────────────────────────────
# Create a sub-model that outputs the first Conv layer's activations

activation_model = models.Model(
    inputs  = model.input,
    outputs = model.get_layer('conv1').output   # shape: (1, H, W, 32)
)

# Load a cat and a dog image
def get_feature_maps(img_path):
    img   = keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    arr   = keras_image.img_to_array(img) / 255.0
    arr   = np.expand_dims(arr, axis=0)
    return img, activation_model.predict(arr, verbose=0)

cat_img, cat_maps = get_feature_maps(
    os.path.join(VALIDATION_CATS_DIR, os.listdir(VALIDATION_CATS_DIR)[0]))
dog_img, dog_maps = get_feature_maps(
    os.path.join(VALIDATION_DOGS_DIR, os.listdir(VALIDATION_DOGS_DIR)[0]))

fig, axes = plt.subplots(4, 9, figsize=(18, 8))
fig.suptitle('Feature maps from Conv Layer 1 — Cat (left) vs Dog (right)', fontsize=12)

# Row 0-1: cat feature maps
axes[0][0].imshow(cat_img.resize((100,100))); axes[0][0].set_title('CAT\noriginal', fontsize=8)
axes[0][0].axis('off')
for i in range(1, 9):
    axes[0][i].imshow(cat_maps[0,:,:,i-1], cmap='viridis')
    axes[0][i].set_title(f'F{i}', fontsize=7); axes[0][i].axis('off')
axes[1][0].axis('off')
for i in range(1, 9):
    axes[1][i].imshow(cat_maps[0,:,:,i+7], cmap='viridis')
    axes[1][i].set_title(f'F{i+8}', fontsize=7); axes[1][i].axis('off')

# Row 2-3: dog feature maps
axes[2][0].imshow(dog_img.resize((100,100))); axes[2][0].set_title('DOG\noriginal', fontsize=8)
axes[2][0].axis('off')
for i in range(1, 9):
    axes[2][i].imshow(dog_maps[0,:,:,i-1], cmap='plasma')
    axes[2][i].set_title(f'F{i}', fontsize=7); axes[2][i].axis('off')
axes[3][0].axis('off')
for i in range(1, 9):
    axes[3][i].imshow(dog_maps[0,:,:,i+7], cmap='plasma')
    axes[3][i].set_title(f'F{i+8}', fontsize=7); axes[3][i].axis('off')

plt.tight_layout()
plt.show()

---

# Step 8 — Prediction

## Theory

At inference time:
1. Load image, resize to 150×150, normalise to [0,1]
2. Add batch dimension: `(150,150,3)` → `(1,150,150,3)`
3. `model.predict()` runs a forward pass with **Dropout disabled**
4. Output is a single float $\hat{y} \in (0,1)$
   - $\hat{y} < 0.5$ → **cat**, confidence = $(1 - \hat{y}) \times 100\%$
   - $\hat{y} \geq 0.5$ → **dog**, confidence = $\hat{y} \times 100\%$

In [ ]:
def predict_image(img_path):
    """
    Full prediction pipeline for a single image.
    Returns prediction ('CAT' or 'DOG'), confidence, and raw sigmoid output.
    """
    # Step 1: load and resize
    img       = keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))

    # Step 2: convert to array and normalise
    img_array = keras_image.img_to_array(img) / 255.0   # (150,150,3)

    # Step 3: add batch dimension
    img_batch = np.expand_dims(img_array, axis=0)        # (1,150,150,3)

    # Step 4: forward pass (Dropout OFF during predict)
    raw = float(model.predict(img_batch, verbose=0)[0][0])

    # Step 5: threshold at 0.5
    if raw >= 0.5:
        return {'label': 'DOG',  'confidence': raw*100,       'raw': raw}
    else:
        return {'label': 'CAT',  'confidence': (1-raw)*100,   'raw': raw}


# Test on 4 cats + 4 dogs
test_images = (
    [os.path.join(VALIDATION_CATS_DIR, f) for f in os.listdir(VALIDATION_CATS_DIR)[:4]] +
    [os.path.join(VALIDATION_DOGS_DIR, f) for f in os.listdir(VALIDATION_DOGS_DIR)[:4]]
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Cat vs Dog Predictions', fontsize=14)

for i, (img_path, ax) in enumerate(zip(test_images, axes.flat)):
    result  = predict_image(img_path)
    true    = 'CAT' if 'cats' in img_path else 'DOG'
    correct = result['label'] == true

    img = keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    ax.imshow(img)
    ax.set_title(
        f"Pred: {result['label']}  ({result['confidence']:.1f}%)\n"
        f"True: {true}  {'✓' if correct else '✗'}",
        fontsize=10,
        color='green' if correct else 'red'
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Save the model ────────────────────────────────────────────────────────────
model.save('cat_dog_cnn_final.keras')
print('✓ Model saved to: cat_dog_cnn_final.keras')
print()
print('To reload anywhere:')
print('  import tensorflow as tf')
print('  model = tf.keras.models.load_model("cat_dog_cnn_final.keras")')

---

# Summary

## What you built

A 4-block CNN that classifies cat and dog photos with ~85% accuracy on 2,000 training images.

## Key concepts mastered

| Concept | What it does | Why it matters |
|---|---|---|
| **Convolution** | Sliding dot-product with learned filters | Detects local spatial patterns |
| **Filters** | Small weight matrices (e.g. 3×3) | Each filter detects one type of pattern |
| **ReLU** | `max(0, x)` | Introduces non-linearity |
| **MaxPooling** | Keep max in 2×2 window | Translation invariance + size reduction |
| **Dropout** | Randomly zero activations | Prevents overfitting |
| **Data augmentation** | Random flips, rotations, zooms | Artificially expands training set |
| **Sigmoid output** | Maps to (0,1) | Probability for binary classification |
| **Binary cross-entropy** | Penalises wrong probabilities | Correct loss for 2-class problems |

## Key maths

| Formula | Name |
|---|---|
| $Z[i,j,f] = \sum_{m,n,c} W[m,n,c,f] \cdot X[i+m,j+n,c] + b$ | Convolution |
| $A = \max(0, Z)$ | ReLU |
| $P[i,j] = \max(2{\times}2\text{ window})$ | MaxPooling |
| $\sigma(x) = 1/(1+e^{-x})$ | Sigmoid |
| $L = -[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ | Binary cross-entropy |

## Next steps

- **Transfer learning** — use VGG16/ResNet50 pretrained on ImageNet → get 95%+ accuracy on same 2,000 images
- **Grad-CAM** — visualise which pixels the model focused on to make its decision
- **Full dataset** — train on all 25,000 images from Kaggle
- **Multi-class** — extend to 10 or 100 animal species
- **Deploy** — wrap in a Flask API or Gradio demo

---
*Built as a Computer Vision learning project.*